# Simulated Annealing Feature Selection for BRCA Classification

This notebook loads all four aligned BRCA omics modalities, keeps every feature at the start, and then selects a 50-feature subset with simulated annealing. Each candidate subset is scored with XGBoost on an inner validation split, and the selected subset is evaluated on the outer test fold across 5 folds.

The default data source is the BRCA aligned matrices and the BRCA label file in the workspace.

In [33]:
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from xgboost import XGBClassifier

SEED = 42
FEATURE_BUDGET = 400
OUTER_FOLDS = 5
INNER_VALID_SIZE = 0.2
ANNEAL_STEPS = 100
START_TEMPERATURE = 3.0
COOLING_RATE = 0.92
USE_CUDA = True

np.random.seed(SEED)


def resolve_dataset_file(relative_parts: list[str]) -> Path:
    candidates = [
        Path(*relative_parts),
        Path("Cancer-Multi-Omics-Benchmark", *relative_parts),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find dataset file: {'/'.join(relative_parts)}")


def load_feature_matrix(path: Path, suffix: str) -> pd.DataFrame:
    raw = pd.read_csv(path, index_col=0)
    matrix = raw.T.copy()
    matrix.index = matrix.index.astype(str)
    matrix.columns = [f"{feature}_{suffix}" for feature in matrix.columns.astype(str)]
    matrix = matrix.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return matrix


def load_labels(path: Path) -> pd.Series:
    labels = pd.read_csv(path).iloc[:, 0].astype(int)
    labels.index = pd.RangeIndex(len(labels))
    return labels


def load_all_aligned_features() -> pd.DataFrame:
    aligned_files = [
        ("BRCA_mRNA_aligned.csv", "mRNA"),
        ("BRCA_miRNA_aligned.csv", "miRNA"),
        ("BRCA_CNV_aligned.csv", "CNV"),
        ("BRCA_Methy_aligned.csv", "Methy"),
    ]
    feature_frames = []
    base_path = ["Main_Dataset", "Classification_datasets", "GS-BRCA", "Aligned"]
    for file_name, suffix in aligned_files:
        feature_frames.append(load_feature_matrix(resolve_dataset_file(base_path + [file_name]), suffix))
    combined = pd.concat(feature_frames, axis=1)
    combined = combined.loc[:, ~combined.columns.duplicated()].copy()
    return combined


def xgb_version_tuple() -> tuple[int, int]:
    major, minor, *_ = (int(part) for part in xgb.__version__.split(".") if part.isdigit())
    return major, minor


def make_xgb_classifier(random_state: int, use_cuda: bool = USE_CUDA) -> XGBClassifier:
    params = {
        "n_estimators": 400,
        "max_depth": 7,
        "learning_rate": 0.05,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "min_child_weight": 1,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "objective": "multi:softprob",
        "num_class": 5,
        "eval_metric": "mlogloss",
        "random_state": random_state,
        "n_jobs": -1,
        "verbosity": 0,
    }

    if use_cuda:
        if xgb_version_tuple() >= (2, 0):
            params["tree_method"] = "hist"
            params["device"] = "cuda"
        else:
            params["tree_method"] = "gpu_hist"
            params["predictor"] = "gpu_predictor"
    else:
        params["tree_method"] = "hist"

    return XGBClassifier(**params)

In [34]:
DATA_FILE = None
LABEL_FILE = resolve_dataset_file([
    "Main_Dataset",
    "Classification_datasets",
    "GS-BRCA",
    "Aligned",
    "BRCA_label_num.csv",
])

X = load_all_aligned_features()
y = load_labels(LABEL_FILE)

if len(X) != len(y):
    raise ValueError(f"Sample count mismatch: X has {len(X)} rows, labels have {len(y)} rows")

print(f"Loaded matrix: {X.shape}")
print(f"Loaded labels: {y.shape}")
print(f"Classes: {sorted(y.unique().tolist())}")
print(f"Feature example: {X.columns[:5].tolist()}")

Loaded matrix: (671, 34022)
Loaded labels: (671,)
Classes: [0, 1, 2, 3, 4]
Feature example: ['FBXL15_mRNA', 'LOC105372824_mRNA', 'TMEM41B_mRNA', 'CHUK_mRNA', 'ZNF518B_mRNA']


In [35]:
def score_feature_subset(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_valid: pd.DataFrame,
    y_valid: pd.Series,
    features: list[str],
    random_state: int,
) -> tuple[float, XGBClassifier]:
    model = make_xgb_classifier(random_state=random_state)
    model.fit(
        X_train[features],
        y_train,
        eval_set=[(X_valid[features], y_valid)],
        verbose=False,
    )
    predictions = model.predict(X_valid[features])
    macro_f1 = f1_score(y_valid, predictions, average="macro")
    return macro_f1, model


def mutate_subset(current_subset: list[str], candidate_pool: list[str], rng: np.random.Generator) -> list[str]:
    current_set = set(current_subset)
    drop_feature = rng.choice(current_subset)
    available = [feature for feature in candidate_pool if feature not in current_set]
    add_feature = rng.choice(available)
    next_subset = [feature for feature in current_subset if feature != drop_feature]
    next_subset.append(add_feature)
    return sorted(next_subset)


def simulated_annealing_feature_search(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_valid: pd.DataFrame,
    y_valid: pd.Series,
    candidate_pool: list[str],
    subset_size: int = FEATURE_BUDGET,
    steps: int = ANNEAL_STEPS,
    start_temperature: float = START_TEMPERATURE,
    cooling_rate: float = COOLING_RATE,
    seed: int = SEED,
) -> tuple[list[str], float, list[dict[str, float]]]:
    if len(candidate_pool) < subset_size:
        raise ValueError(
            f"Candidate pool has {len(candidate_pool)} features, which is smaller than the requested subset size {subset_size}."
        )

    rng = np.random.default_rng(seed)
    current_subset = sorted(rng.choice(candidate_pool, size=subset_size, replace=False).tolist())
    current_score, _ = score_feature_subset(X_train, y_train, X_valid, y_valid, current_subset, seed)
    best_subset = current_subset.copy()
    best_score = current_score
    temperature = float(start_temperature)
    history: list[dict[str, float]] = []

    for step in range(steps):
        proposal_subset = mutate_subset(current_subset, candidate_pool, rng)
        proposal_score, _ = score_feature_subset(
            X_train,
            y_train,
            X_valid,
            y_valid,
            proposal_subset,
            seed + step + 1,
        )

        delta = proposal_score - current_score
        accept = delta >= 0 or rng.random() < math.exp(delta / max(temperature, 1e-12))
        if accept:
            current_subset = proposal_subset
            current_score = proposal_score

        if proposal_score > best_score:
            best_subset = proposal_subset
            best_score = proposal_score

        history.append(
            {
                "step": step + 1,
                "temperature": temperature,
                "proposal_f1": proposal_score,
                "current_f1": current_score,
                "best_f1": best_score,
            }
        )
        temperature *= cooling_rate

    return best_subset, best_score, history

In [36]:
outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED)
fold_results = []
feature_frequency: dict[str, int] = {}
start_time = time.time()

for fold_number, (train_index, test_index) in enumerate(outer_cv.split(X, y), start=1):
    X_train_outer = X.iloc[train_index].reset_index(drop=True)
    X_test_outer = X.iloc[test_index].reset_index(drop=True)
    y_train_outer = y.iloc[train_index].reset_index(drop=True)
    y_test_outer = y.iloc[test_index].reset_index(drop=True)

    X_train_inner, X_valid_inner, y_train_inner, y_valid_inner = train_test_split(
        X_train_outer,
        y_train_outer,
        test_size=INNER_VALID_SIZE,
        random_state=SEED + fold_number,
        stratify=y_train_outer,
    )

    candidate_pool = X_train_outer.columns.tolist()
    best_subset, best_valid_f1, history = simulated_annealing_feature_search(
        X_train_inner,
        y_train_inner,
        X_valid_inner,
        y_valid_inner,
        candidate_pool=candidate_pool,
        subset_size=FEATURE_BUDGET,
        steps=ANNEAL_STEPS,
        seed=SEED + fold_number,
    )

    final_model = make_xgb_classifier(random_state=SEED + fold_number)
    final_model.fit(X_train_outer[best_subset], y_train_outer, verbose=False)
    test_predictions = final_model.predict(X_test_outer[best_subset])

    fold_result = {
        "fold": fold_number,
        "valid_macro_f1": best_valid_f1,
        "test_accuracy": accuracy_score(y_test_outer, test_predictions),
        "test_balanced_accuracy": balanced_accuracy_score(y_test_outer, test_predictions),
        "test_macro_f1": f1_score(y_test_outer, test_predictions, average="macro"),
        "selected_feature_count": len(best_subset),
        "selected_features": ",".join(best_subset),
        "anneal_steps": len(history),
    }
    fold_results.append(fold_result)

    for feature in best_subset:
        feature_frequency[feature] = feature_frequency.get(feature, 0) + 1

    print(
        f"Fold {fold_number}: valid macro F1={best_valid_f1:.4f}, "
        f"test macro F1={fold_result['test_macro_f1']:.4f}, "
        f"test accuracy={fold_result['test_accuracy']:.4f}"
    )

results_df = pd.DataFrame(fold_results)
print(f"\nCompleted in {(time.time() - start_time) / 60:.2f} minutes")
results_df

Fold 1: valid macro F1=0.7556, test macro F1=0.8279, test accuracy=0.8667
Fold 2: valid macro F1=0.8072, test macro F1=0.6542, test accuracy=0.7910
Fold 3: valid macro F1=0.7700, test macro F1=0.6766, test accuracy=0.8134
Fold 4: valid macro F1=0.6766, test macro F1=0.7464, test accuracy=0.8507
Fold 5: valid macro F1=0.7489, test macro F1=0.6378, test accuracy=0.7687

Completed in 74.99 minutes


,fold,valid_macro_f1,test_accuracy,test_balanced_accuracy,test_macro_f1,selected_feature_count,selected_features,anneal_steps
0,1,0.755596,0.866667,0.771218,0.827942,400,"ACADSB_Methy,ACR_mRNA,ACTL10_CNV,ACTR8_mRNA,AD...",100
1,2,0.807238,0.791045,0.622195,0.654175,400,"ABCA10_mRNA,ACTA1_CNV,ADAM20_CNV,ADAMTS19_Meth...",100
2,3,0.770006,0.813433,0.627189,0.676630,400,"ABCF3_mRNA,ACPT_Methy,ADAM22_CNV,ADRA2B_CNV,AK...",100
3,4,0.676641,0.850746,0.713065,0.746435,400,"ABHD14B_CNV,ABHD3_mRNA,ACSM4_CNV,ACTL8_mRNA,AC...",100
4,5,0.748901,0.768657,0.593527,0.637812,400,"AADAT_CNV,ABCA10_mRNA,ABCA3_mRNA,ABCF3_mRNA,AB...",100


In [37]:
summary = results_df.agg(
    {
        "valid_macro_f1": ["mean", "std"],
        "test_accuracy": ["mean", "std"],
        "test_balanced_accuracy": ["mean", "std"],
        "test_macro_f1": ["mean", "std"],
    }
)

print("\nFold metrics summary")
print(summary)

feature_frequency_df = (
    pd.Series(feature_frequency, name="fold_count")
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "feature"})
)

print("\nTop recurring features across folds")
print(feature_frequency_df.head(20))

results_df.to_csv("simulated_annealing_fold_metrics.csv", index=False)
feature_frequency_df.to_csv("simulated_annealing_feature_frequency.csv", index=False)

results_df, feature_frequency_df.head(20)


Fold metrics summary
      valid_macro_f1  test_accuracy  test_balanced_accuracy  test_macro_f1
mean        0.751676       0.818109                0.665439       0.708599
std         0.047635       0.040691                0.074098       0.078532

Top recurring features across folds
                 feature  fold_count
0              SOCS5_CNV           2
1            DHX40_Methy           2
2          KLHDC8B_Methy           2
3              DNMBP_CNV           2
4             APOL3_mRNA           2
5           ZBTB38_Methy           2
6             ZNF484_CNV           2
7   hsa.mir.320b.2_miRNA           2
8            NUP210_mRNA           2
9            ZNF516_mRNA           2
10           SMAD5_Methy           2
11           JAZF1_Methy           2
12             ATXN2_CNV           2
13          KCNIP4_Methy           2
14           PADI3_Methy           2
15             UPF3B_CNV           2
16        KIAA1958_Methy           2
17            BEX1_Methy           2
18           

(   fold  valid_macro_f1  test_accuracy  test_balanced_accuracy  test_macro_f1  \
 0     1        0.755596       0.866667                0.771218       0.827942   
 1     2        0.807238       0.791045                0.622195       0.654175   
 2     3        0.770006       0.813433                0.627189       0.676630   
 3     4        0.676641       0.850746                0.713065       0.746435   
 4     5        0.748901       0.768657                0.593527       0.637812   
 
    selected_feature_count                                  selected_features  \
 0                     400  ACADSB_Methy,ACR_mRNA,ACTL10_CNV,ACTR8_mRNA,AD...   
 1                     400  ABCA10_mRNA,ACTA1_CNV,ADAM20_CNV,ADAMTS19_Meth...   
 2                     400  ABCF3_mRNA,ACPT_Methy,ADAM22_CNV,ADRA2B_CNV,AK...   
 3                     400  ABHD14B_CNV,ABHD3_mRNA,ACSM4_CNV,ACTL8_mRNA,AC...   
 4                     400  AADAT_CNV,ABCA10_mRNA,ABCA3_mRNA,ABCF3_mRNA,AB...   
 
    anneal_steps  